**Experimentation**

In [7]:
import pandas as pd
df=pd.read_csv("../logs.csv")
df.head()

,log_text,time_to_failure,high_severity
0,2026-03-02 14:44:33 INFO [service] Health chec...,138,0
1,2026-03-02 14:47:33 INFO [service] User login ...,98,0
2,2026-03-02 14:48:33 ERROR [service] NullPointe...,10,1
3,2026-03-02 14:53:33 WARN [service] Slow respon...,36,0
4,2026-03-02 14:56:33 INFO [service] Scheduled j...,137,0


Building Preprocessing Function

In [4]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def clean_log(text):
    # Remove timestamps
    text = re.sub(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", "", text)
    
    # Remove IP addresses (future safe)
    text = re.sub(r"\d+\.\d+\.\d+\.\d+", "", text)
    
    # Remove brackets
    text = re.sub(r"\[.*?\]", "", text)
    
    # Lowercase
    text = text.lower()
    
    # Process with spaCy
    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if not token.is_stop and not token.is_punct and token.is_alpha:
            tokens.append(token.lemma_)
    
    return " ".join(tokens)

Apply Cleaning

In [5]:
df["clean_text"] = df["log_text"].apply(clean_log)

df[["log_text", "clean_text"]].head()

,log_text,clean_text
0,2026-03-02 14:44:33 INFO [service] Health chec...,info health check pass
1,2026-03-02 14:47:33 INFO [service] User login ...,info user login successful
2,2026-03-02 14:48:33 ERROR [service] NullPointe...,error nullpointerexception module x
3,2026-03-02 14:53:33 WARN [service] Slow respon...,warn slow response detect api
4,2026-03-02 14:56:33 INFO [service] Scheduled j...,info schedule job complete


TF-IDF Vectorization

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=500)

X = vectorizer.fit_transform(df["clean_text"])

X.shape

(200, 45)

Checking the feature names

In [7]:
vectorizer.get_feature_names_out()[:20]

array(['api', 'application', 'cache', 'capacity', 'check', 'complete',
       'configuration', 'connection', 'crash', 'database', 'detect',
       'disk', 'error', 'exhaust', 'health', 'high', 'increase', 'info',
       'job', 'load'], dtype=object)

Train/Test split

In [ ]:
from sklearn.model_selection import train_test_split

y_reg = df["time_to_failure"]
y_clf = df["high_severity"]

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf,
    test_size=0.2,
    random_state=42
)

Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
reg_model=LinearRegression()
reg_model.fit(X_train,y_reg_train)

Evaluate Regression Model

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error

y_reg_train_pred=reg_model.predict(X_train)
y_reg_test_pred=reg_model.predict(X_test)

print("Train R2:", r2_score(y_reg_train, y_reg_train_pred))
print("Test R2:", r2_score(y_reg_test, y_reg_test_pred))

print("Train MAE:", mean_absolute_error(y_reg_train, y_reg_train_pred))
print("Test MAE:", mean_absolute_error(y_reg_test, y_reg_test_pred))

Classification - Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_model = LogisticRegression(max_iter=1000)
clf_model.fit(X_train, y_clf_train)

Evaluate Classification

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

y_clf_train_pred = clf_model.predict(X_train)
y_clf_test_pred = clf_model.predict(X_test)

print("Train Accuracy:", accuracy_score(y_clf_train, y_clf_train_pred))
print("Test Accuracy:", accuracy_score(y_clf_test, y_clf_test_pred))

print("Precision:", precision_score(y_clf_test, y_clf_test_pred))
print("Recall:", recall_score(y_clf_test, y_clf_test_pred))

print("\nClassification Report:\n")
print(classification_report(y_clf_test, y_clf_test_pred))

Cross Validation(Avoid Overfitting)

In [ ]:
from sklearn.model_selection import cross_val_score

reg_cv = cross_val_score(reg_model, X, y_reg, cv=5, scoring="r2")
print("Regression CV Mean R2:", reg_cv.mean())

clf_cv = cross_val_score(clf_model, X, y_clf, cv=5, scoring="accuracy")
print("Classification CV Mean Accuracy:", clf_cv.mean())

In [9]:
import pandas as pd
print(df["high_severity"].value_counts())

high_severity
0    108
1     92
Name: count, dtype: int64
